# EPS Dynamic-GNN: results exploration
Run the pipeline scripts first (`src/data/generate_dataset.py`, `src/train.py` x4, `src/evaluate.py`, `src/interpretability.py`), then use this notebook to explore the generated dataset and results interactively.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from src.data.dataset import EPSWindowDataset, load_metadata

meta = load_metadata('../data/processed')
print(meta['fault_classes'])
print(meta['split_counts'])

## Dataset class balance

In [ ]:
counts = meta['split_counts']
classes = meta['fault_classes']
splits = list(counts.keys())
fig, ax = plt.subplots(figsize=(8,4))
x = np.arange(len(classes))
w = 0.25
for i, s in enumerate(splits):
    vals = [counts[s].get(c, 0) for c in classes]
    ax.bar(x + i*w, vals, width=w, label=s)
ax.set_xticks(x+w); ax.set_xticklabels(classes, rotation=20, ha='right')
ax.legend(); ax.set_ylabel('samples'); ax.set_title('Class balance by split')
plt.show()

## Performance table

In [ ]:
import pandas as pd
perf = pd.read_csv('../results/metrics/performance_table.csv')
perf

## Training curves

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for model in ['cnn_baseline','lstm_baseline','static_gnn','dynamic_gnn']:
    path = f'../results/metrics/{model}_history.json'
    if not os.path.exists(path):
        continue
    h = json.load(open(path))['history']
    val_acc = [e['val_acc'] for e in h]
    ax.plot(val_acc, label=model)
ax.set_xlabel('epoch'); ax.set_ylabel('val accuracy'); ax.legend(); ax.set_title('Validation accuracy by model')
plt.show()

## Figures (confusion matrices, attention-by-class, dynamic adjacency)

In [ ]:
from IPython.display import Image, display
for f in ['confusion_matrices.png', 'attention_by_class.png', 'dynamic_adjacency_eclipse_transition.png']:
    p = f'../results/figures/{f}'
    if os.path.exists(p):
        display(Image(p))